In [ ]:
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

In [ ]:
with open('config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

В этой теме мы поработаем с данными, посвященными определению рака молочной железы на основе различных признаков анализа клеток в биопсии (радиус, кривизна, симметрия). Известно, что этот датасет линейно разделим.

In [ ]:
df = pd.read_csv('data.csv')
df.head()

In [ ]:
df.info()

**Задание**: Проведите краткий EDA. Есть ли выбросы в данных, какие столбцы коррелируют больше всего, стоит ли преобразоывавть какие-то признаки? Хватит 3-4 графиков или таблиц (но можно больше).

In [ ]:
num_df = df.select_dtypes(exclude=['object'])

In [ ]:
import seaborn as sns

print("Пропуски в данных:\n", df.isnull().sum())

plt.figure(figsize=(10, 6))
df[['radius_mean', 'texture_mean', 'perimeter_mean']].boxplot()
plt.title("Выбросы")
plt.show()

def corrplot(df, method="pearson", annot=True, **kwargs):
    
    fig, ax = plt.subplots(figsize=(10,10)) 
    sns.heatmap(
        df.corr(method),
        vmin=-1.0,
        vmax=1.0,
        cmap="icefire",
        annot=annot,
        ax=ax,
        **kwargs,
    )
corrplot(num_df, annot=None)

plt.figure(figsize=(6, 4))
sns.countplot(x='diagnosis', data=df)
plt.title("Распределение классов (0: Benign, 1: Malignant)")
plt.show()

In [ ]:
df = df.drop(['id', 'Unnamed: 32'], axis=1)
df.head()

In [ ]:
df['diagnosis'] = df['diagnosis'].replace({'B': 0, 'M': 1}).astype(int)
df.head()

**Задание**: выведите, сколько в датасете примеров позитивного и негативного класса.

In [ ]:
print(df['diagnosis'].value_counts())

In [ ]:
target = 'diagnosis'
features = list(df.columns)
features.remove('diagnosis')
features

In [ ]:
X = df[features]
y = df[[target]]

Попробуем обучить логистическую регрессию на этих данных. Обратите внимание, что по умолчанию применяется L2 регуляризация,мы будем строить предсказания без нее. Однако, в качестве упражнения, сравним результаты с масштабированием признаков и без.

**Задание**: оцените, насколько сбалансированы признаки по масштабу. Попробуйте ответить до запуска кода, стоит ли их сначала масштабировать и почему. 

In [ ]:
num_df = df[features]
stats = pd.DataFrame({
    'Среднее': num_df.mean(),
    'Минимум': num_df.min(),
    'Максимум': num_df.max(),
    'Диапазон': num_df.max() - num_df.min()
})

stats_sorted = stats.sort_values('Диапазон', ascending=False)
print("Топ-5 признаков по диапазону значений:")
print(stats_sorted[['Диапазон']].head())

print(f"Наименьший диапазон: {stats['Диапазон'].min():.2f}")
print(f"Наибольший диапазон: {stats['Диапазон'].max():.2f}")
print(f"Отношение наибольшего к наименьшему диапазону: {stats['Диапазон'].max() / stats['Диапазон'].min():.2f}")

Без масштабирования:

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X.values, y.values.reshape(-1), train_size=0.8, shuffle=True)
clf = LogisticRegression(penalty=None)
clf.fit(X_train, y_train)
clf.score(X_test, y_test)

С масштабированием:

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X.values, y.values.reshape(-1), train_size=0.8, shuffle=True)
clf = LogisticRegression(penalty=None)
ss = StandardScaler()
X_train = ss.fit_transform(X_train)
X_test = ss.transform(X_test)
clf.fit(X_train, y_train)
clf.score(X_test, y_test)

Все классификаторы в Sklearn имеют два режима - предсказание лейблов и вероятностей. Предсказание вероятностей дает нам необработанные оценки принадлежности к тому или иному классу. Модель в таком случае возвращает вектор (для каждого семпла) размера N (где N - число классов). 

**Вопрос**: Какого размера будет предсказание в случае бинарной логистической регрессии? А многоклассовой? Другими словами, в каких случаях негативный класс добавляется как отдельный?

In [ ]:
df_results = pd.DataFrame({
    'pred': clf.predict(X_test).reshape(-1),
    'pred_proba': clf.predict_proba(X_test)[:, 1],
    'true': y_test.reshape(-1),
})

**Задание**: Постройте матрицу предсказаний 100x2 для регрессии с двумя классами, где в каждой строке будут случайные значения. 
1) Получите из этого оценку принадлежности к классу с помощью сигмоиды и софтмакса. 
2) Постройте предсказание класса. В случае сигмоиды предсказывайте принадлежность к классу на основе границы, софтмакса - по максимальной вероятности

**Вопрос***: как еще можно предсказать класс? Всегда ли нужно брать именно эти функции?

In [ ]:
np.random.seed(42)
random_scores = np.random.randn(100, 2)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

sigmoid_probs = sigmoid(random_scores[:, 1])
sigmoid_preds = (sigmoid_probs >= 0.5).astype(int)

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

softmax_probs = softmax(random_scores)
softmax_preds = np.argmax(softmax_probs, axis=1)

df_random = pd.DataFrame({
    'sigmoid_prob': sigmoid_probs,
    'sigmoid_pred': sigmoid_preds,
    'softmax_prob_class1': softmax_probs[:, 1],
    'softmax_pred': softmax_preds
})
print(df_random.head())

In [ ]:
df_results.head(20)

# Метрики классификации


## Метрики на основе лейблов
Рассмотрим, какие у нас могут быть тезультаты классификации.

* TP (true positive) - правильно предсказали: рак есть, что модель и предсказала
* FP (false positive) - неправильно предсказали: рака нет,  а модель предсказала, что есть (1st order error)
* FN (false negative) - неправильно предсказали: рак вообще-то есть,  а модель предсказала, что нет (2nd order error)!
* TN (true negative) - правильно предсказали: рака нет, что модель и предсказала


Pos/Neg - общее количество объектов класса 1/0

Метрики:

* $ \text{Accuracy} = \frac{TP + TN}{Pos+Neg}$ - Доля правильных ответов
* $ \text{Error rate} = 1 -\text{accuracy}$ - Доля ошибок
* $ \text{Precision} =\frac{TP}{TP + FP}$ - Точность
* $ \text{Recall} =\frac{TP}{TP + FN} = \frac{TP}{Pos}$ - Полнота
* $ \text{F}_\beta \text{-score} = (1 + \beta^2) \cdot \frac{\mathrm{precision} \cdot \mathrm{recall}}{(\beta^2 \cdot \mathrm{precision}) + \mathrm{recall}}$ F-мера (часто используют F1-меру, где $\beta=1$)

### ROC кривая

ROC кривая измеряет насколько хорошо классификатор разделяет два класса. Она построена на предсказании вероятности. Площадь под ней (ROC-AUC) является неплохой оценкой общего качества предсказаний. 
 
Пусть $y_{\rm i}$ - истинная метрка и $\hat{y}_{\rm i}$ - прогноз вероятности для $i^{\rm th}$ объекта.

Число положительных и отрицательных объектов: $\mathcal{I}_{\rm 1} = \{i: y_{\rm i}=1\}$ and $\mathcal{I}_{\rm 0} = \{i: y_{\rm i}=0\}$.

Для каждого порогового значения вероятности $\tau$ считаем True Positive Rate (TPR) и False Positive Rate (FPR):

\begin{equation}
TPR(\tau) = \frac{1}{I_{\rm 1}} \sum_{i \in \mathcal{I}_{\rm 1}} I[\hat{y}_{\rm i} \ge \tau] = \frac{TP(\tau)}{TP(\tau) + FN(\tau)} = \frac{TP(\tau)}{Pos}
\end{equation}

\begin{equation}
FPR(\tau) = \frac{1}{I_{\rm 0}} \sum_{i \in \mathcal{I}_{\rm 0}} I[\hat{y}_{\rm i} \ge \tau]= \frac{FP(\tau)}{FP(\tau) + TN(\tau)} = \frac{FP(\tau)}{Neg}
\end{equation}

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(
    n_samples=10000, n_features=10, n_informative=5, n_redundant=5, random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

Используем для сравнения случайный предикт. Иногда это не худшая стратегия. Если в данных мало сигнала, случайное предсказание может работать лучше ложного.

In [ ]:
from sklearn.dummy import DummyClassifier
random_classifier = DummyClassifier(strategy='uniform', random_state=42).fit(X_train, y_train)
y_random = random_classifier.predict_proba(X_test)[:,1]
y_random

In [ ]:
random_preds = random_classifier.predict(X_test)
random_preds

In [ ]:
from sklearn.metrics import average_precision_score

from sklearn.metrics import precision_recall_curve
from sklearn.metrics import PrecisionRecallDisplay

from sklearn.metrics import roc_auc_score
from sklearn.metrics import RocCurveDisplay

def depict_pr_roc(y_true, y_pred, classifier_name='Some Classifier', ax=None):
    if ax is None:
        fig, ax = plt.subplots(1, 2, figsize=(11, 5))

    print(classifier_name, 'metrics')
    PrecisionRecallDisplay.from_predictions(y_true, y_pred, ax=ax[0], name=classifier_name)
    print('AUC-PR: %.4f' % average_precision_score(y_true, y_pred))
    ax[0].set_title("PRC")
    ax[0].set_ylim(0, 1.1)

    RocCurveDisplay.from_predictions(y_true, y_pred, ax=ax[1], name=classifier_name)
    print('AUC-ROC: %.4f' % roc_auc_score(y_true, y_pred))
    ax[1].set_title("ROC")
    ax[1].set_ylim(0, 1.1)

    plt.tight_layout()
    plt.legend()


depict_pr_roc(y_test, y_random, 'Random Classifier')

Также посчитаем другие метрики на основе лейблов.

**Задание:** Дополните код по рассчету метрик.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def quality_metrics_report(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    tn = np.sum((y_true == 0) & (y_pred == 0))

    accuracy = accuracy_score(y_true, y_pred)
    error_rate = 1 - accuracy
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    return [tp, fp, fn, tn, accuracy, error_rate, precision, recall, f1]

In [ ]:
# dataframe для сравнения
# методов классификации по метрикам
df_metrics = pd.DataFrame(
    columns=['acc', 'er', 'precision', 'recall', 'f1', 'auc_pr', 'roc_auc_score', 'reg_const']
)
precision, recall, _ = precision_recall_curve(y_test, y_random)
# добавление очередной строки с характеристиками метода
[tp, fp, fn, tn, accuracy, error_rate, precision, recall, f1] = quality_metrics_report(y_test, random_preds)
df_metrics.loc['Random Classifier'] = [
      accuracy, error_rate, precision, recall, f1,
      average_precision_score(y_test, y_random),
      roc_auc_score(y_test, y_random),
      0,
]

# по аналогии результаты следующих экспериментов можно будет собрать в табличку
df_metrics

In [ ]:
clf = LogisticRegression()
clf.fit(X_train, y_train)
clf.score(X_test, y_test)

In [ ]:
clf = LogisticRegression(penalty=None, max_iter=1000)
clf.fit(X_train, y_train)
lr_preds = clf.predict(X_test)
lr_probs = clf.predict_proba(X_test)[:, 1]

[tp, fp, fn, tn, accuracy, error_rate, precision, recall, f1] = quality_metrics_report(y_test, lr_preds)

df_metrics.loc['Logistic Regression'] = [
    accuracy, error_rate, precision, recall, f1,
    average_precision_score(y_test, lr_probs),
    roc_auc_score(y_test, lr_probs), 0
]

df_metrics

Согласуются ли метрики? В чем может быть проблема accuracy?

**Задание**: Соберите табличку для разных классификаторов.

**Задание**: Постройте график PR-curve, ROC-curve для лучшего из них

In [ ]:
best_classifier = 'Logistic Regression (L2)' if df_metrics.loc['Logistic Regression (L2)', 'f1'] > df_metrics.loc['Logistic Regression (No Reg)', 'f1'] else 'Logistic Regression (No Reg)'
best_probs = lr_probs if best_classifier == 'Logistic Regression (L2)' else lr_probs

fig, ax = plt.subplots(1, 2, figsize=(11, 5))
depict_pr_roc(y_test, best_probs, classifier_name=best_classifier, ax=ax)
plt.show()

**Задание:** Постройте таблицу точности для набора данных wbdc. Сделайте по таблице метрик на обучающей и тестовой выборках. В таблице сравните разные преобразования признаков и гиперпараметры (регуляризацию). Можно сделать три-четыре эксперимента. 
- На каком эксперименте получилось достичь лучшего качества на трейне?
- А на тесте?
- Переобучается ли модель?

In [ ]:
from sklearn.metrics import accuracy_score, average_precision_score, roc_auc_score

df_metrics_wbdc = pd.DataFrame(
    columns=['acc_train', 'acc_test', 'precision_test', 'recall_test', 'f1_test', 'auc_pr_test', 'roc_auc_test', 'description']
)

X = df[features].values
y = df[target].values.ravel()

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=42)
clf = LogisticRegression(penalty=None, max_iter=1000)
clf.fit(X_train, y_train)
train_preds = clf.predict(X_train)
test_preds = clf.predict(X_test)
test_probs = clf.predict_proba(X_test)[:, 1]
[tp, fp, fn, tn, acc_test, er, prec_test, rec_test, f1_test] = quality_metrics_report(y_test, test_preds)
df_metrics_wbdc.loc['No Scaling, No Reg'] = [
    accuracy_score(y_train, train_preds), acc_test, prec_test, rec_test, f1_test,
    average_precision_score(y_test, test_probs), roc_auc_score(y_test, test_probs), 'No Scaling, penalty=None'
]

ss = StandardScaler()
X_train_scaled = ss.fit_transform(X_train)
X_test_scaled = ss.transform(X_test)
clf.fit(X_train_scaled, y_train)
train_preds = clf.predict(X_train_scaled)
test_preds = clf.predict(X_test_scaled)
test_probs = clf.predict_proba(X_test_scaled)[:, 1]
[tp, fp, fn, tn, acc_test, er, prec_test, rec_test, f1_test] = quality_metrics_report(y_test, test_preds)
df_metrics_wbdc.loc['Scaling, No Reg'] = [
    accuracy_score(y_train, train_preds), acc_test, prec_test, rec_test, f1_test,
    average_precision_score(y_test, test_probs), roc_auc_score(y_test, test_probs), 'Scaling, penalty=None'
]

clf_l2 = LogisticRegression(penalty='l2', C=1.0, max_iter=1000)
clf_l2.fit(X_train_scaled, y_train)
train_preds = clf_l2.predict(X_train_scaled)
test_preds = clf_l2.predict(X_test_scaled)
test_probs = clf_l2.predict_proba(X_test_scaled)[:, 1]
[tp, fp, fn, tn, acc_test, er, prec_test, rec_test, f1_test] = quality_metrics_report(y_test, test_preds)
df_metrics_wbdc.loc['Scaling, L2 C=1.0'] = [
    accuracy_score(y_train, train_preds), acc_test, prec_test, rec_test, f1_test,
    average_precision_score(y_test, test_probs), roc_auc_score(y_test, test_probs), 'Scaling, penalty=l2, C=1.0'
]

clf_l2 = LogisticRegression(penalty='l2', C=0.1, max_iter=1000)
clf_l2.fit(X_train_scaled, y_train)
train_preds = clf_l2.predict(X_train_scaled)
test_preds = clf_l2.predict(X_test_scaled)
test_probs = clf_l2.predict_proba(X_test_scaled)[:, 1]
[tp, fp, fn, tn, acc_test, er, prec_test, rec_test, f1_test] = quality_metrics_report(y_test, test_preds)
df_metrics_wbdc.loc['Scaling, L2 C=0.1'] = [
    accuracy_score(y_train, train_preds), acc_test, prec_test, rec_test, f1_test,
    average_precision_score(y_test, test_probs), roc_auc_score(y_test, test_probs), 'Scaling, penalty=l2, C=0.1'
]

print(df_metrics_wbdc)

best_train = df_metrics_wbdc['acc_train'].idxmax()
best_test = df_metrics_wbdc['f1_test'].idxmax()
print(f"Лучшее качество на train: {best_train}")
print(f"Лучшее качество на test (по F1): {best_test}")
print(f"Переобучение (разница acc_train - acc_test): {df_metrics_wbdc.loc[best_train, 'acc_train'] - df_metrics_wbdc.loc[best_train, 'acc_test']:.4f}")